### **Step 1:** Find the cholestrol lab type and match it to the encounter. The diagnostic table will tell us the effective date for the completed report, but we need to match which encounter contains a lab completed for cholesterol.

In [28]:
import pandas
import duckdb

connection = duckdb.connect('novellia.duckdb')
connection.execute('''
SELECT
    DISTINCT observation_name
FROM
    stg_observation
WHERE
    category = 'laboratory'
    AND LOWER(observation_name) LIKE '%chol%'
''').df()

,observation_name
0,Cholesterol [Mass/volume] in Serum or Plasma
1,Low Density Lipoprotein Cholesterol
2,Cholesterol in HDL [Mass/volume] in Serum or P...


In [29]:
import pandas
import duckdb

connection = duckdb.connect('novellia.duckdb')
connection.execute('''
SELECT
    *
FROM
    stg_diagnosticreport
''').df()

,diagnostic_report_id,status,report_name,report_code,report_code_system,patient_id,encounter_id,effective_date_time,issued,performer
0,000010da-ddbf-f134-9e23-ff11e5a99656,final,None,34117-2,http://loinc.org,7a57ff65-7378-1ac9-be86-7e12f51f94e3,a98b29b0-f9d5-1c1a-0109-cb17e128994c,2021-10-31T15:14:15-04:00,2021-10-31T15:14:15.258-04:00,Dr. Sancho742 Pérez790
1,0000492b-baa5-c467-aaa1-d5c5d8504fea,final,CBC panel - Blood by Automated count,58410-2,http://loinc.org,c7c5c028-f1fb-962b-8db2-dfd4c6d6b02a,969c91ac-c3a7-8940-56c7-5b87c8f3aa0f,2022-06-10T19:33:30-04:00,2022-06-10T19:33:30.023-04:00,GRACEMED HEALTH CLINIC INC
2,000052d6-0c00-572b-20ad-2b1050c8258c,final,Alcohol Use Disorder Identification Test - Con...,72109-2,http://loinc.org,a8fb9909-8833-a4d1-67a4-c6bd5398ca92,9ac36bde-ae72-053e-14bd-dfab61292658,1982-04-19T18:15:05-05:00,1982-04-19T18:15:05.549-05:00,None
3,000062d5-5ee9-9585-7d15-7b23db812059,final,Urinalysis macro (dipstick) panel - Urine,24357-6,http://loinc.org,dda3d07b-ecb4-1aa8-62ac-541027559e11,2b04527b-d707-b265-d00d-577b43cbaf42,1994-08-15T15:12:09-04:00,1994-08-15T15:12:09.623-04:00,ASCENSION VIA CHRISTI HOSPITAL PITTSBURG INC
4,00010cdc-2198-84ed-8d27-9d4eb10ab634,final,Patient Health Questionnaire 2 item (PHQ-2) [R...,55757-9,http://loinc.org,776451bb-0662-cea7-3691-afe5ec755aff,192cab16-5c5e-c35b-f097-33437ffcca79,2022-04-27T01:36:54-04:00,2022-04-27T01:36:54.142-04:00,None
...,...,...,...,...,...,...,...,...,...,...
160344,fffe4b58-8f7b-ec15-253f-ca46b7452ed6,final,None,34117-2,http://loinc.org,4a326793-814f-5274-8c12-22b85873b2e6,80fce9ca-78fa-dda9-8457-5dfd1adab7bd,2005-12-05T04:07:53-05:00,2005-12-05T04:07:53.829-05:00,Dr. Ronald408 Emard19
160345,fffe7aec-a0b0-6538-a27d-219269c82073,final,Urinalysis macro (dipstick) panel - Urine,24357-6,http://loinc.org,e3cb004a-0888-6270-0dcb-90b1ef9eae0a,3041be97-1124-c20c-890b-8396e67c935a,2018-04-15T21:56:27-04:00,2018-04-15T21:56:27.914-04:00,WESLEY MEDICAL CENTER
160346,ffff031a-590e-6f0a-7963-1a7b8a048147,final,None,34117-2,http://loinc.org,c3375230-4466-aa3b-8e33-4a8dd87a53b9,7fd19fc5-2c2a-d5a7-ffd1-c4fedfea73b8,2023-02-24T07:23:47-05:00,2023-02-24T07:23:47.509-05:00,Dr. Adan632 Hilpert278
160347,ffff1dcf-c1ee-8f79-7196-9a84888e9f91,final,Patient Health Questionnaire 2 item (PHQ-2) [R...,55757-9,http://loinc.org,e4e994c7-c1df-9821-e664-f4e36b90709b,50af6f12-e615-cd6b-9135-d7370c0b1520,2002-11-05T05:30:27-05:00,2002-11-05T05:30:27.419-05:00,None


### **Step 2:** JOIN diagnostic to observation so that I can find the correct / latest report data for completed cholestrol lab per patient. Find all cholesterol labs per patient done, but only pick the most recently completed one.

In [30]:
connection.execute('''
WITH latest_cholesterol_encounter AS (
    SELECT
        DISTINCT patient_id,
        MAX_BY(encounter_id, effective_date_time) AS latest_cholesterol_encounter
    FROM
        stg_observation
    WHERE
        category = 'laboratory'
        AND LOWER(observation_name) LIKE '%chol%'
    GROUP BY
        patient_id
)
SELECT
    DISTINCT a.patient_id,
    MAX_BY(a.diagnostic_report_id, effective_date_time) AS latest_report_with_chol_lab
FROM
    stg_diagnosticreport a
JOIN
    latest_cholesterol_encounter b
    ON a.patient_id = b.patient_id
    AND a.encounter_id = b.latest_cholesterol_encounter
GROUP BY
    1
''').df()

,patient_id,latest_report_with_chol_lab
0,c68a041d-dea4-5d65-e7dd-40f598e27762,c71893ee-3703-83cc-c400-9f9b384d6ea6
1,3c5b7d07-e021-31bc-630c-63629b5f63f4,2dc410a0-c40c-39e9-bc9c-27a645ada8b3
2,74e72b8b-a3cf-f6f4-b652-dcbf8536216b,d94264bc-444d-8516-ce0a-9c28d4ec0152
3,8e57f080-9fc3-2e67-072f-635dcd659397,b048fcb5-7783-72f3-94a3-e0390261387d
4,69500434-f89a-c35b-30bb-3ed8d08c5bc3,54dfab33-6781-be4a-6285-fd7001913013
...,...,...
687,c75187b9-42ca-9dbd-26ba-d90f402459ae,50c8cb9c-a9c0-72d6-b42b-dfa8b9c3bdbe
688,be53a07a-f057-39de-768e-f9ba821c4775,50665350-0748-cb59-22fe-20b3e7d14111
689,0b052286-9534-99a8-8d5e-06c2c04a7df7,ebb09e29-e68d-88fc-bcb8-de8f72196f85
690,341a43b1-8d26-55a3-267e-653cffeafa36,400c3553-5599-085d-0cef-d0f84052ae4a


### **ANSWER 04** Table above showing the latest reports for the given criteria.

### **ANSWER 05** I'm going to take a stab at exploring a patient with the most # of immunizations. Apparently, this person has LESS than the average # of observations per patient as well as LESS than the # of diagnostic reports per patient. Could this be because they are well immunized and therefore, more healthy and in need of less robust testing/results?

In [41]:
connection.execute('''
WITH patient_scope AS (
    SELECT
        patient_id,
        COUNT(immunization_id) AS immunizations
    FROM
       stg_immunization
    GROUP BY 1
    ORDER BY 2 DESC
    LIMIT 1
)
SELECT
    obs.*
FROM
    stg_observation obs
JOIN
    patient_scope ps
    ON obs.patient_id = ps.patient_id
ORDER BY effective_date_time DESC
''').df()

,observation_id,status,category,observation_name,observation_code,observation_code_system,patient_id,encounter_id,effective_date_time,issued,value_quantity,value_unit,value_codeable_concept,value_string
0,772922ab-b564-c0f2-91ce-2d7b9ea35a61,final,vital-signs,Body temperature,8310-5,http://loinc.org,ef309281-d86d-8a67-ff74-0b9edc15d67f,9e126a1c-a3f2-beb4-a367-4ba57ac951e0,2023-02-26T17:50:09-05:00,2023-02-26T17:50:09.200-05:00,37.810,Cel,None,None
1,5f2650b6-0b45-94f8-ae70-01a5fadc10cb,final,social-history,Tobacco smoking status,72166-2,http://loinc.org,ef309281-d86d-8a67-ff74-0b9edc15d67f,636a9877-4af4-004c-9112-19ec2c859fe4,2023-02-19T17:50:09-05:00,2023-02-19T17:50:09.200-05:00,NaN,None,Never smoked tobacco (finding),None
2,67c2651f-547d-1259-8556-287613399cd2,final,vital-signs,Pain severity - 0-10 verbal numeric rating [Sc...,72514-3,http://loinc.org,ef309281-d86d-8a67-ff74-0b9edc15d67f,636a9877-4af4-004c-9112-19ec2c859fe4,2023-02-19T17:50:09-05:00,2023-02-19T17:50:09.200-05:00,0.000,{score},None,None
3,779aeb92-9a27-54c5-868c-2f76d697977b,final,vital-signs,Body Weight,29463-7,http://loinc.org,ef309281-d86d-8a67-ff74-0b9edc15d67f,636a9877-4af4-004c-9112-19ec2c859fe4,2023-02-19T17:50:09-05:00,2023-02-19T17:50:09.200-05:00,29.500,kg,None,None
4,4745782e-5c80-466a-ebee-dc3342d95c6d,final,vital-signs,Body Height,8302-2,http://loinc.org,ef309281-d86d-8a67-ff74-0b9edc15d67f,636a9877-4af4-004c-9112-19ec2c859fe4,2023-02-19T17:50:09-05:00,2023-02-19T17:50:09.200-05:00,138.200,cm,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
285,241db697-5691-aac2-03e2-d84025fc7930,final,laboratory,Hematocrit [Volume Fraction] of Blood by Autom...,4544-3,http://loinc.org,ef309281-d86d-8a67-ff74-0b9edc15d67f,a8814f46-0673-3b7a-f1a1-db4021fe5faa,2013-04-21T18:50:09-04:00,2013-04-21T18:50:09.200-04:00,37.496,%,None,None
286,180fc6d0-4701-c9c9-fed1-c8befd2a4129,final,social-history,Tobacco smoking status,72166-2,http://loinc.org,ef309281-d86d-8a67-ff74-0b9edc15d67f,a8814f46-0673-3b7a-f1a1-db4021fe5faa,2013-04-21T18:50:09-04:00,2013-04-21T18:50:09.200-04:00,NaN,None,Never smoked tobacco (finding),None
287,05ad7b4a-6915-abec-3cbc-d08e2e389419,final,laboratory,Platelets [#/volume] in Blood by Automated count,777-3,http://loinc.org,ef309281-d86d-8a67-ff74-0b9edc15d67f,a8814f46-0673-3b7a-f1a1-db4021fe5faa,2013-04-21T18:50:09-04:00,2013-04-21T18:50:09.200-04:00,296.910,10*3/uL,None,None
288,270d106a-e773-9fd8-2c46-5624bd7cd7a8,final,laboratory,Hemoglobin [Mass/volume] in Blood,718-7,http://loinc.org,ef309281-d86d-8a67-ff74-0b9edc15d67f,a8814f46-0673-3b7a-f1a1-db4021fe5faa,2013-04-21T18:50:09-04:00,2013-04-21T18:50:09.200-04:00,16.496,g/dL,None,None


In [43]:
connection.execute('''
WITH patient_scope AS (
    SELECT
        patient_id,
        COUNT(immunization_id) AS immunizations
    FROM
       stg_immunization
    GROUP BY 1
    ORDER BY 2 DESC
    LIMIT 1
)
SELECT
    diag.*
FROM
    stg_diagnosticreport diag
JOIN
    patient_scope ps
    ON diag.patient_id = ps.patient_id
ORDER BY effective_date_time DESC
''').df()

,diagnostic_report_id,status,report_name,report_code,report_code_system,patient_id,encounter_id,effective_date_time,issued,performer
0,5b1fb3fa-7b76-3631-ff9e-7fcca100214c,final,None,34117-2,http://loinc.org,ef309281-d86d-8a67-ff74-0b9edc15d67f,9e126a1c-a3f2-beb4-a367-4ba57ac951e0,2023-02-26T17:50:09-05:00,2023-02-26T17:50:09.200-05:00,Dr. Theo630 Hammes673
1,4e97677e-b85a-16c0-2bbb-5ddd49866619,final,None,34117-2,http://loinc.org,ef309281-d86d-8a67-ff74-0b9edc15d67f,636a9877-4af4-004c-9112-19ec2c859fe4,2023-02-19T17:50:09-05:00,2023-02-19T17:50:09.200-05:00,Dr. Theo630 Hammes673
2,5015145f-63dd-3e1d-d6e0-e684edc859c1,final,None,34117-2,http://loinc.org,ef309281-d86d-8a67-ff74-0b9edc15d67f,de5e0114-70d4-a4a2-068a-ec97ff38c13b,2023-02-12T17:50:09-05:00,2023-02-12T17:50:09.200-05:00,Dr. Quintin944 Murphy561
3,e99f32ff-d83e-b9d4-5e2f-0c7a1ceff738,final,None,34117-2,http://loinc.org,ef309281-d86d-8a67-ff74-0b9edc15d67f,ae1f53f8-2fde-849f-f50f-bfbaf17777f4,2022-04-24T18:50:09-04:00,2022-04-24T18:50:09.200-04:00,Dr. Phung894 Williamson769
4,349a5f6f-9cd5-8407-c695-8eb832fd3075,final,None,34117-2,http://loinc.org,ef309281-d86d-8a67-ff74-0b9edc15d67f,c093d870-eb56-5499-6135-dc8687aec65c,2022-04-10T18:50:09-04:00,2022-04-10T18:50:09.200-04:00,Dr. Quintin944 Murphy561
5,fe54c9ab-a196-7d61-5ce5-f5fafa73a158,final,None,34117-2,http://loinc.org,ef309281-d86d-8a67-ff74-0b9edc15d67f,56c22fa6-e4e9-f63e-e445-58d3f047fd27,2022-04-01T04:50:09-04:00,2022-04-01T04:50:09.200-04:00,Dr. Theo630 Hammes673
6,d007a6ec-cbc6-459a-ad2b-9d4b5cf294f3,final,None,34117-2,http://loinc.org,ef309281-d86d-8a67-ff74-0b9edc15d67f,636063d0-5b91-72f5-5673-a7020bbed45b,2021-06-09T21:50:09-04:00,2021-06-09T21:50:09.200-04:00,Dr. Theo630 Hammes673
7,a5bffaf7-bc90-3839-2004-ac352004c813,final,None,34117-2,http://loinc.org,ef309281-d86d-8a67-ff74-0b9edc15d67f,7d7fc6d6-81e1-8613-23a9-de34eaaffb8b,2021-04-18T18:50:09-04:00,2021-04-18T18:50:09.200-04:00,Dr. Phung894 Williamson769
8,1b089297-234c-7b84-d82a-eb35d506c3f0,final,None,34117-2,http://loinc.org,ef309281-d86d-8a67-ff74-0b9edc15d67f,97af1e36-f14c-97df-26e3-04b504dda3c3,2020-04-12T18:50:09-04:00,2020-04-12T18:50:09.200-04:00,Dr. Phung894 Williamson769
9,9bdd5644-be88-cda2-fa6e-82941765f198,final,None,34117-2,http://loinc.org,ef309281-d86d-8a67-ff74-0b9edc15d67f,442e3b16-0304-dc77-2111-fe16c0f49dcc,2020-03-15T18:50:09-04:00,2020-03-15T18:50:09.200-04:00,Dr. Theo630 Hammes673


In [40]:
connection.execute('''
WITH per_patient_count AS (
    SELECT
        patient_id,
        COUNT(observation_id) AS observations
    FROM
        stg_observation
    GROUP BY
        1
)
SELECT
    AVG(observations)
FROM
    per_patient_count
''').df()

,avg(observations)
0,606.226399


In [44]:
connection.execute('''
WITH per_patient_count AS (
    SELECT
        patient_id,
        COUNT(diagnostic_report_id) AS reports
    FROM
        stg_diagnosticreport
    GROUP BY
        1
)
SELECT
    AVG(reports)
FROM
    per_patient_count
''').df()

,avg(reports)
0,140.16521
